In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
df_jan = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet")
df_feb = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet")

In [21]:
df_jan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3066766 entries, 0 to 3066765
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[ns]
 2   tpep_dropoff_datetime  datetime64[ns]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  airport_fee           

## Q1: 19 columns

In [4]:
# January
df_jan['duration'] = df_jan.tpep_dropoff_datetime - df_jan.tpep_pickup_datetime
df_jan.duration = df_jan.duration.apply(lambda td: td.total_seconds() / 60)


# February
df_feb['duration'] = df_feb.tpep_dropoff_datetime - df_feb.tpep_pickup_datetime
df_feb.duration = df_feb.duration.apply(lambda td: td.total_seconds() / 60)


In [5]:
df_jan.duration.describe()

count    3.066766e+06
mean     1.566900e+01
std      4.259435e+01
min     -2.920000e+01
25%      7.116667e+00
50%      1.151667e+01
75%      1.830000e+01
max      1.002918e+04
Name: duration, dtype: float64

## Q2: 42.59

In [6]:
df_jan_no_outlier = df_jan[(df_jan['duration'] <= 60) & (df_jan['duration'] >= 1)]
df_feb_no_outlier = df_feb[(df_feb['duration'] <= 60) & (df_feb['duration'] >= 1)]

len(df_jan_no_outlier) / len(df_jan)

0.9812202822125979

## Q3: 98%

In [7]:
df_jan['trip_distance']

0          0.97
1          1.10
2          2.51
3          1.90
4          1.43
           ... 
3066761    3.05
3066762    5.80
3066763    4.67
3066764    3.15
3066765    2.85
Name: trip_distance, Length: 3066766, dtype: float64

In [8]:
df_jan_no_outlier[['PULocationID', 'DOLocationID']].dtypes

PULocationID    int64
DOLocationID    int64
dtype: object

In [9]:
categorical = ['PULocationID', 'DOLocationID']
df_jan_no_outlier[categorical] = df_jan_no_outlier[categorical].astype(str)
df_feb_no_outlier[categorical] = df_feb_no_outlier[categorical].astype(str)



/tmp/ipykernel_23137/4150317990.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_jan_no_outlier[categorical] = df_jan_no_outlier[categorical].astype(str)
/tmp/ipykernel_23137/4150317990.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_feb_no_outlier[categorical] = df_feb_no_outlier[categorical].astype(str)


In [10]:
train_dicts = df_jan_no_outlier[categorical].to_dict(orient= 'record')
val_dicts = df_feb_no_outlier[categorical].to_dict(orient = 'record')

/tmp/ipykernel_23137/3023269773.py:1: FutureWarning: Using short name for 'orient' is deprecated. Only the options: ('dict', list, 'series', 'split', 'records', 'index') will be used in a future version. Use one of the above to silence this warning.
  train_dicts = df_jan_no_outlier[categorical].to_dict(orient= 'record')
/tmp/ipykernel_23137/3023269773.py:2: FutureWarning: Using short name for 'orient' is deprecated. Only the options: ('dict', list, 'series', 'split', 'records', 'index') will be used in a future version. Use one of the above to silence this warning.
  val_dicts = df_feb_no_outlier[categorical].to_dict(orient = 'record')


In [11]:
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_train.shape

(3009173, 515)

## Q4: 515

In [14]:
target = 'duration'
y_train = df_jan_no_outlier[target].values
y_val = df_feb_no_outlier[target].values

In [15]:
lr = LinearRegression()
lr.fit(X_train, y_train)


LinearRegression()

In [16]:
y_pred_train = lr.predict(X_train)
rmse_train = mean_squared_error(y_train, y_pred_train, squared= False)
print('rmse on train data= ', rmse_train)

rmse on train data=  7.649261027792376


## Q5: 7.64

In [17]:
y_pred_val = lr.predict(X_val)
rmse_val = mean_squared_error(y_val, y_pred_val, squared= False)
print('rmse on validation data= ', rmse_val)

rmse on validation data=  7.811832836304415


## Q6: 7.81